# Seattle Airbnb — Data Cleaning & Integration

**DSBA 6156 Applied Machine Learning | Fall 2026 | Dzmitry Prybysh**

This notebook implements data cleaning, type conversions, neighborhood alignment, and multi-source integration for the Seattle Airbnb project, building directly upon the data profiling findings documented in `notebooks/01_data_integration_eda.ipynb`.

## Notebook Roadmap & Deliverables
1. **Setup & Environment:** Directory paths, display parameters, and dependency verification.
2. **Raw Data Ingestion & Baseline Verification:** Ingest all five verified CSV sources and confirm shapes against D1 baseline.
3. **Data Cleaning and Type Conversion Functions:** Reusable, vectorized transformations for currency, percentage, boolean, and date fields with unit validation tests.
4. **Cleaning & Type Transformation Execution:** Construct decoupled working copies (`*_clean`) applying type conversions while preserving raw datasets.
5. **Verification & Quality Audit:** Audit non-null preservation (0 parse errors, 100% positional match), document exclusion candidates (`license`, `square_feet`), verify calendar consistency, and summarize validation results.
6. **Neighborhood Alignment & Walk Score Mapping (D2.2):** Exact neighborhood name matches are used (73 neighborhoods); the 14 non-matching neighborhoods are classified as 'No confirmed correspondence' because remaining correspondences are not verified, with many-to-one join verification.
7. **Integrated Dataset Assembly (D2.3):** In-memory construction of Listing-Level View (with Walk Score and review aggregates) and Calendar-Day Panel View (with pre-aggregated events), post-join validation checks, and leakage documentation.
8. **Integration Validation (D2.4):** Five focused verification checks (Rows & Keys, Data Preservation, Walk Score Fidelity, Reviews Concordance, Calendar & Events Alignment) reported in a single compact validation summary table.
9. **Data Export & Documentation (D2.5):** Export processed integrated datasets to `data/processed/` in CSV format, perform full readback verification with type restoration, and document reloading schemas for downstream workflows.


## 1. Setup & Environment

Load project dependencies, configure display parameters, and set project directory paths.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Configure display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)

# Establish robust project paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

assert DATA_DIR.is_dir(), f'Data directory not found: {DATA_DIR}'


## 2. Raw Data Ingestion & Baseline Verification

In this step, we ingest all five source datasets profiled during Phase D1 (`01_data_integration_eda.ipynb`). We assert file existence, verify baseline shapes against our data inventory, and create immutable raw references (`*_raw`) to ensure all downstream cleaning operates strictly on decoupled working copies.


In [2]:
# Load all five raw datasets
listings_raw = pd.read_csv(DATA_DIR / 'listings.csv', low_memory=False)
calendar_raw = pd.read_csv(DATA_DIR / 'calendar.csv', low_memory=False)
reviews_raw = pd.read_csv(DATA_DIR / 'reviews.csv', low_memory=False)
walkscore_raw = pd.read_csv(DATA_DIR / 'seattle_walkscore_raw.csv')
holidays_raw = pd.read_csv(DATA_DIR / 'seattle_holidays_events_2016.csv')

# Verified baseline shapes from Phase D1
EXPECTED_SHAPES = {
    'listings': (3818, 92),
    'calendar': (1393570, 4),
    'reviews': (84849, 6),
    'walkscore': (80, 7),
    'holidays': (19, 5),
}

inventory = []
for name, df, filename in [
    ('listings', listings_raw, 'listings.csv'),
    ('calendar', calendar_raw, 'calendar.csv'),
    ('reviews', reviews_raw, 'reviews.csv'),
    ('walkscore', walkscore_raw, 'seattle_walkscore_raw.csv'),
    ('holidays', holidays_raw, 'seattle_holidays_events_2016.csv'),
]:
    expected = EXPECTED_SHAPES[name]
    actual = df.shape
    assert actual == expected, f'Shape mismatch for {name}: expected {expected}, got {actual}'
    inventory.append({
        'Dataset': name,
        'File': filename,
        'Rows': f'{actual[0]:,}',
        'Columns': actual[1],
        'Memory (MB)': f'{df.memory_usage(deep=True).sum() / (1024**2):.2f}',
        'Status': 'Verified vs D1 Baseline'
    })

df_inventory = pd.DataFrame(inventory)
display(df_inventory)


,Dataset,File,Rows,Columns,Memory (MB),Status
0,listings,listings.csv,"3,818",92,27.33,Verified vs D1 Baseline
1,calendar,calendar.csv,"1,393,570",4,219.02,Verified vs D1 Baseline
2,reviews,reviews.csv,"84,849",6,47.20,Verified vs D1 Baseline
3,walkscore,seattle_walkscore_raw.csv,80,7,0.02,Verified vs D1 Baseline
4,holidays,seattle_holidays_events_2016.csv,19,5,0.01,Verified vs D1 Baseline


## 3. Data Cleaning and Type Conversion Functions

To ensure reproducibility and avoid code duplication across datasets, we define specialized, vectorized transformation functions for four non-numeric data representations:
1. **Currency (`clean_currency`):** Strips currency symbols (`$`), thousands separators (`,`), and whitespace, coercing valid numeric values to `float64` while strictly preserving `NaN` values.
2. **Percentage (`clean_percentage`):** Strips the percentage symbol (`%`) and whitespace, converting values to `float64` on a 0–100 scale while strictly preserving `NaN` values.
3. **Nullable Boolean (`clean_boolean`):** Maps string indicators `'t'` and `'f'` to `True` and `False` using pandas nullable `boolean` dtype (`pd.BooleanDtype()`). This avoids coercing missing values to `False` (which would corrupt data semantics) or falling back to un-vectorized `object` arrays.
4. **Date Parsing (`clean_date`):** Formats string representations into `datetime64[ns]` using strict ISO format (`%Y-%m-%d`), validating temporal validity without introducing synthetic defaults.

Each function includes unit assertion tests on synthetic edge cases prior to dataset-wide application.


In [3]:
def clean_currency(series: pd.Series) -> pd.Series:
    """Strip currency symbols ('$'), commas, and whitespace; convert to float64 preserving NaNs."""
    return (
        series.astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
        .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
        .astype(float)
    )

def clean_percentage(series: pd.Series) -> pd.Series:
    """Strip '%' symbol and whitespace; convert to float64 (0-100 scale) preserving NaNs."""
    return (
        series.astype(str)
        .str.replace('%', '', regex=False)
        .str.strip()
        .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
        .astype(float)
    )

def clean_boolean(series: pd.Series) -> pd.Series:
    """Map 't'/'f' to True/False using pandas nullable boolean dtype, strictly preserving missing values as pd.NA."""
    return series.map({'t': True, 'f': False}).astype('boolean')

def clean_date(series: pd.Series, date_format: str = '%Y-%m-%d') -> pd.Series:
    """Parse date strings into datetime64[ns] with exact format validation, preserving NaNs."""
    return pd.to_datetime(series, format=date_format, errors='coerce')

# Unit tests on synthetic edge-case series
test_curr = pd.Series(['$1,250.00', ' $45.50 ', '$0.00', np.nan, None, ''])
cleaned_curr = clean_currency(test_curr)
assert cleaned_curr.iloc[0] == 1250.0 and cleaned_curr.iloc[1] == 45.5 and cleaned_curr.iloc[2] == 0.0
assert cleaned_curr.isna().sum() == 3

test_pct = pd.Series(['100%', ' 95% ', '0%', np.nan, None])
cleaned_pct = clean_percentage(test_pct)
assert cleaned_pct.iloc[0] == 100.0 and cleaned_pct.iloc[1] == 95.0 and cleaned_pct.iloc[2] == 0.0
assert cleaned_pct.isna().sum() == 2

test_bool = pd.Series(['t', 'f', np.nan, None])
cleaned_bool = clean_boolean(test_bool)
assert cleaned_bool.iloc[0] == True and cleaned_bool.iloc[1] == False
assert pd.isna(cleaned_bool.iloc[2]) and pd.isna(cleaned_bool.iloc[3])
assert str(cleaned_bool.dtype) == 'boolean'

test_dt = pd.Series(['2016-01-04', '2016-12-31', np.nan])
cleaned_dt = clean_date(test_dt)
assert str(cleaned_dt.dtype).startswith('datetime64')
assert cleaned_dt.isna().sum() == 1

print('All unit validation tests for cleaning functions passed successfully.')


All unit validation tests for cleaning functions passed successfully.


## 4. Cleaning & Type Transformation Execution

We now apply our validated transformation functions to create clean working copies:
- `listings_clean`: Currency (6 fields), percentages (2 fields), booleans (9 fields), and dates (5 fields).
- `calendar_clean`: Date (`date`), availability indicator (`available`), and price (`price`).
- `reviews_clean`: Review publication date (`date`).
- `holidays_clean`: Holiday/event calendar date (`date`).
- `walkscore_clean`: Validation of existing integer score metrics (`walk_score`, `transit_score`, `bike_score`, `population`).

**Strict Boundary Conditions Observed:**
- Zero rows dropped across all tables.
- Zero columns dropped (exclusion candidates `license` and `square_feet` are retained and profiled in Section 5).
- Zero missing values imputed (no synthetic imputation of Walk Score, ratings, or prices).
- Zero cross-table joins or premature listing-level aggregations (preventing data leakage).


In [4]:
# Create decoupled working copies
listings_clean = listings_raw.copy()
calendar_clean = calendar_raw.copy()
reviews_clean = reviews_raw.copy()
holidays_clean = holidays_raw.copy()
walkscore_clean = walkscore_raw.copy()

# 1. Listings Transformations
CURRENCY_COLS_LISTINGS = [
    'price', 'weekly_price', 'monthly_price',
    'security_deposit', 'cleaning_fee', 'extra_people'
]
PERCENTAGE_COLS_LISTINGS = [
    'host_response_rate', 'host_acceptance_rate'
]
BOOLEAN_COLS_LISTINGS = [
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
    'is_location_exact', 'has_availability', 'requires_license',
    'instant_bookable', 'require_guest_profile_picture', 'require_guest_phone_verification'
]
DATE_COLS_LISTINGS = [
    'last_scraped', 'host_since', 'calendar_last_scraped',
    'first_review', 'last_review'
]

for col in CURRENCY_COLS_LISTINGS:
    listings_clean[col] = clean_currency(listings_raw[col])

for col in PERCENTAGE_COLS_LISTINGS:
    listings_clean[col] = clean_percentage(listings_raw[col])

for col in BOOLEAN_COLS_LISTINGS:
    listings_clean[col] = clean_boolean(listings_raw[col])

for col in DATE_COLS_LISTINGS:
    listings_clean[col] = clean_date(listings_raw[col])

# 2. Calendar Transformations
calendar_clean['date'] = clean_date(calendar_raw['date'])
calendar_clean['available'] = clean_boolean(calendar_raw['available'])
calendar_clean['price'] = clean_currency(calendar_raw['price'])

# 3. Reviews Transformations
reviews_clean['date'] = clean_date(reviews_raw['date'])

# 4. Holidays & Events Transformations
holidays_clean['date'] = clean_date(holidays_raw['date'])

# 5. Walk Score Validation
WALKSCORE_NUMERIC_COLS = ['rank', 'walk_score', 'transit_score', 'bike_score', 'population']
for col in WALKSCORE_NUMERIC_COLS:
    assert pd.api.types.is_integer_dtype(walkscore_clean[col]), f'{col} in walkscore is not integer!'

print('Transformations successfully applied to all working copies.')


Transformations successfully applied to all working copies.


## 5. Verification & Quality Audit

We perform a multi-dimensional validation audit on all converted columns:
1. **Non-Null Preservation & Positional Alignment:** Verify that the null count after conversion exactly matches the null count before conversion (`Null Delta == 0`), and confirm element-wise that missing values remain in the exact same rows.
2. **Range & Type Verification:** Confirm that data types match expected analytical representations and numeric ranges align with empirical domain boundaries.
3. **Calendar Availability vs. Price Consistency:** Empirically verify row-level alignment between unobserved prices and unavailable calendar dates (`available == 'f'`).
4. **Exclusion Candidates Documentation:** Formally audit `license` and `square_feet` to document reasons for eventual exclusion without dropping them prematurely.
5. **Boundary Checks:** Verify row counts, column counts, and absence of premature data manipulation.

### 5.1 Type Conversion Audit


In [5]:
# Construct comprehensive before-and-after conversion verification table
audit_records = []

def audit_field(dataset_name, col_name, semantic_type, raw_s, clean_s):
    raw_nulls = int(raw_s.isna().sum())
    clean_nulls = int(clean_s.isna().sum())
    null_delta = clean_nulls - raw_nulls
    assert null_delta == 0, f'Parse error in {dataset_name}.{col_name}: null count increased by {null_delta}!'
    
    # Element-wise positional null match assertion
    assert (raw_s.isna() == clean_s.isna()).all(), f'Positional null mismatch in {dataset_name}.{col_name}!'

    if pd.api.types.is_datetime64_any_dtype(clean_s):
        valid_dates = clean_s.dropna()
        val_range = f"{valid_dates.min().strftime('%Y-%m-%d')} to {valid_dates.max().strftime('%Y-%m-%d')}"
    elif pd.api.types.is_bool_dtype(clean_s) or str(clean_s.dtype) == 'boolean':
        t_cnt = int((clean_s == True).sum())
        f_cnt = int((clean_s == False).sum())
        val_range = f'True: {t_cnt:,} | False: {f_cnt:,}'
    elif pd.api.types.is_numeric_dtype(clean_s):
        valid_vals = clean_s.dropna()
        val_range = f'[{valid_vals.min():.2f}, {valid_vals.max():.2f}] (mean: {valid_vals.mean():.2f})'
    else:
        val_range = 'N/A'

    audit_records.append({
        'Dataset': dataset_name,
        'Column': col_name,
        'Semantic Type': semantic_type,
        'Raw Dtype': str(raw_s.dtype),
        'Clean Dtype': str(clean_s.dtype),
        'Raw Nulls': raw_nulls,
        'Clean Nulls': clean_nulls,
        'Null Delta': null_delta,
        'Null Position Match': '100% Exact',
        'Value Range / Distribution': val_range
    })

# Listings currency
for c in CURRENCY_COLS_LISTINGS:
    audit_field('listings', c, 'Currency ($)', listings_raw[c], listings_clean[c])

# Listings percentage
for c in PERCENTAGE_COLS_LISTINGS:
    audit_field('listings', c, 'Percentage (%)', listings_raw[c], listings_clean[c])

# Listings boolean
for c in BOOLEAN_COLS_LISTINGS:
    audit_field('listings', c, 'Boolean (t/f)', listings_raw[c], listings_clean[c])

# Listings dates
for c in DATE_COLS_LISTINGS:
    audit_field('listings', c, 'Date (YYYY-MM-DD)', listings_raw[c], listings_clean[c])

# Calendar fields
audit_field('calendar', 'date', 'Date (YYYY-MM-DD)', calendar_raw['date'], calendar_clean['date'])
audit_field('calendar', 'available', 'Boolean (t/f)', calendar_raw['available'], calendar_clean['available'])
audit_field('calendar', 'price', 'Currency ($)', calendar_raw['price'], calendar_clean['price'])

# Reviews date
audit_field('reviews', 'date', 'Date (YYYY-MM-DD)', reviews_raw['date'], reviews_clean['date'])

# Holidays date
audit_field('holidays', 'date', 'Date (YYYY-MM-DD)', holidays_raw['date'], holidays_clean['date'])

df_audit = pd.DataFrame(audit_records)
display(df_audit)

# Row-level verification between available == False and missing price in calendar
cal_f_and_price_na = int(((calendar_clean['available'] == False) & (calendar_clean['price'].isna())).sum())
cal_f_and_price_valid = int(((calendar_clean['available'] == False) & (calendar_clean['price'].notna())).sum())
cal_t_and_price_na = int(((calendar_clean['available'] == True) & (calendar_clean['price'].isna())).sum())
cal_t_and_price_valid = int(((calendar_clean['available'] == True) & (calendar_clean['price'].notna())).sum())

assert cal_f_and_price_valid == 0, 'Found unavailable dates with valid prices!'
assert cal_t_and_price_na == 0, 'Found available dates with missing prices!'

assert (df_audit['Null Delta'] == 0).all(), 'Found non-zero null deltas during type conversion!'
print('Validation Result: 100% of converted fields preserved valid entries with 0 parse errors and exact row-level null positions.')
print(f'Calendar Row-Level Alignment: 100% of {cal_f_and_price_na:,} missing prices occur strictly on unavailable dates (available == False). Zero misaligned rows.')


Validation Result: 100% of converted fields preserved valid entries with 0 parse errors and exact row-level null positions.
Calendar Row-Level Alignment: 100% of 459,028 missing prices occur strictly on unavailable dates (available == False). Zero misaligned rows.


,Dataset,Column,Semantic Type,Raw Dtype,Clean Dtype,Raw Nulls,Clean Nulls,Null Delta,Null Position Match,Value Range / Distribution
0,listings,price,Currency ($),str,float64,0,0,0,100% Exact,"[20.00, 1000.00] (mean: 127.98)"
1,listings,weekly_price,Currency ($),str,float64,1809,1809,0,100% Exact,"[100.00, 6300.00] (mean: 788.48)"
2,listings,monthly_price,Currency ($),str,float64,2301,2301,0,100% Exact,"[500.00, 19500.00] (mean: 2613.34)"
3,listings,security_deposit,Currency ($),str,float64,1952,1952,0,100% Exact,"[95.00, 5000.00] (mean: 287.61)"
4,listings,cleaning_fee,Currency ($),str,float64,1030,1030,0,100% Exact,"[5.00, 300.00] (mean: 61.71)"
5,listings,extra_people,Currency ($),str,float64,0,0,0,100% Exact,"[0.00, 300.00] (mean: 10.67)"
6,listings,host_response_rate,Percentage (%),str,float64,523,523,0,100% Exact,"[17.00, 100.00] (mean: 94.89)"
7,listings,host_acceptance_rate,Percentage (%),str,float64,773,773,0,100% Exact,"[0.00, 100.00] (mean: 99.97)"
8,listings,host_is_superhost,Boolean (t/f),str,boolean,2,2,0,100% Exact,"True: 778 | False: 3,038"
9,listings,host_has_profile_pic,Boolean (t/f),str,boolean,2,2,0,100% Exact,"True: 3,809 | False: 7"


### 5.2 Audit of Column Exclusion Candidates & Analytical Justification

In accordance with project governance rules, columns are **never dropped automatically without empirical justification**. Here we formally examine two primary exclusion candidates identified during D1 profiling: `license` and `square_feet`.


In [6]:
# Detailed audit of exclusion candidates
exclusion_candidates = [
    {
        'Column': 'license',
        'Dataset': 'listings',
        'Total Rows': len(listings_clean),
        'Missing Count': int(listings_clean['license'].isna().sum()),
        'Missing Pct': f"{listings_clean['license'].isna().mean():.2%}",
        'Non-Null Count': int(listings_clean['license'].notna().sum()),
        'Status': 'RETAINED (Exclusion Candidate)',
        'Analytical Rationale': '100% missing (3,818 records). The column contains no usable information for the current analysis and is a candidate for exclusion.'
    },
    {
        'Column': 'square_feet',
        'Dataset': 'listings',
        'Total Rows': len(listings_clean),
        'Missing Count': int(listings_clean['square_feet'].isna().sum()),
        'Missing Pct': f"{listings_clean['square_feet'].isna().mean():.2%}",
        'Non-Null Count': int(listings_clean['square_feet'].notna().sum()),
        'Status': 'RETAINED (Exclusion Candidate)',
        'Analytical Rationale': '97.46% missing (only 97 non-null listings). Severe missingness prevents general use. Retained for now; final exclusion decision will be documented during feature preparation.'
    }
]

df_exclusion = pd.DataFrame(exclusion_candidates)
display(df_exclusion)

# Check constant and near-constant columns for documentation
constant_checks = [
    {
        'Column': 'has_availability',
        'Value Distribution': str(dict(listings_clean['has_availability'].value_counts(dropna=False))),
        'Commentary': 'Constant feature (100% True). Zero variance across all 3,818 listings.'
    },
    {
        'Column': 'requires_license',
        'Value Distribution': str(dict(listings_clean['requires_license'].value_counts(dropna=False))),
        'Commentary': 'Constant feature (100% False). Zero variance across all 3,818 listings.'
    },
    {
        'Column': 'host_acceptance_rate',
        'Value Distribution': str(dict(listings_clean['host_acceptance_rate'].value_counts(dropna=False).head(3))),
        'Commentary': 'Near-constant feature (3,044 listings are 100%, 1 listing is 0%, 773 are NaN).'
    }
]

df_constant = pd.DataFrame(constant_checks)
display(df_constant)


,Column,Dataset,Total Rows,Missing Count,Missing Pct,Non-Null Count,Status,Analytical Rationale
0,license,listings,3818,3818,100.00%,0,RETAINED (Exclusion Candidate),"100% missing (3,818 records). The column contains no usable information for the current analysis and is a candidate for exclusion."
1,square_feet,listings,3818,3721,97.46%,97,RETAINED (Exclusion Candidate),97.46% missing (only 97 non-null listings). Severe missingness prevents general use. Retained for now; final exclusion decision will be documented during feature preparation.


,Column,Value Distribution,Commentary
0,has_availability,{np.True_: np.int64(3818)},"Constant feature (100% True). Zero variance across all 3,818 listings."
1,requires_license,{np.False_: np.int64(3818)},"Constant feature (100% False). Zero variance across all 3,818 listings."
2,host_acceptance_rate,"{100.0: np.int64(3044), nan: np.int64(773), 0.0: np.int64(1)}","Near-constant feature (3,044 listings are 100%, 1 listing is 0%, 773 are NaN)."


### 5.3 D2.1 Validation Summary

To ensure strict compliance with workflow constraints prior to proceeding to D2.2:
1. **Row Count Preservation:**
   - `listings`: exactly 3,818 rows (0 rows dropped).
   - `calendar`: exactly 1,393,570 rows (0 rows dropped).
   - `reviews`: exactly 84,849 rows (0 rows dropped).
   - `walkscore`: exactly 80 rows (0 rows dropped).
   - `holidays`: exactly 19 rows (0 rows dropped).
2. **Column Count Preservation:**
   - `listings`: exactly 92 columns (0 columns dropped).
   - `calendar`: exactly 4 columns (0 columns dropped).
   - `reviews`: exactly 6 columns (0 columns dropped).
   - `walkscore`: exactly 7 columns (0 columns dropped).
   - `holidays`: exactly 5 columns (0 columns dropped).
3. **Imputation Status:**
   - Missing values across all DataFrames remain strictly unmodified (`NaN`/`<NA>`).
   - No Walk Score imputation or synthetic neighborhood estimation performed.
   - Review scores rating missing values (647 listings / 16.95%) remain un-imputed.
4. **Isolation & Leakage Prevention:**
   - Calendar and Reviews datasets remain independent at their native temporal granularity (`date`).
   - No target-dependent features or temporal aggregations have been created yet. Leakage checks will be required after target definition.
   - No cross-table joins performed.


In [7]:
# Formal assertion of integrity boundaries
assert len(listings_clean) == 3818 and listings_clean.shape[1] == 92, 'Listings shape altered!'
assert len(calendar_clean) == 1393570 and calendar_clean.shape[1] == 4, 'Calendar shape altered!'
assert len(reviews_clean) == 84849 and reviews_clean.shape[1] == 6, 'Reviews shape altered!'
assert len(walkscore_clean) == 80 and walkscore_clean.shape[1] == 7, 'Walk Score shape altered!'
assert len(holidays_clean) == 19 and holidays_clean.shape[1] == 5, 'Holidays shape altered!'

print('=== D2.1 VALIDATION RESULTS ===')
print('✓ All 5 working copies created and decoupled from raw inputs.')
print('✓ 100% row preservation across all tables (0 rows dropped).')
print('✓ 100% column preservation across all tables (0 columns dropped).')
print('✓ Zero missing value imputation performed (all NaNs preserved).')
print('✓ Zero cross-table merges performed.')
print('✓ No target-dependent features or temporal aggregations created yet (leakage checks deferred to target definition).')
print('✓ D2.1 Data Cleaning and Type Conversion complete and ready for review.')


=== D2.1 VALIDATION RESULTS ===
✓ All 5 working copies created and decoupled from raw inputs.
✓ 100% row preservation across all tables (0 rows dropped).
✓ 100% column preservation across all tables (0 columns dropped).
✓ Zero missing value imputation performed (all NaNs preserved).
✓ Zero cross-table merges performed.
✓ No target-dependent features or temporal aggregations created yet (leakage checks deferred to target definition).
✓ D2.1 Data Cleaning and Type Conversion complete and ready for review.


## 6. Neighborhood Alignment & Walk Score Mapping (D2.2)

During Phase D1, comparison between `listings.csv` (`neighbourhood_cleansed`) and `seattle_walkscore_raw.csv` (`neighborhood`) revealed that while 73 neighborhood names match character-for-character (representing 2,733 listings, 71.58%), **14 neighborhood names representing 1,085 listings (28.42%) have no exact string match** in the local 80-row Walk Score dataset.

**Methodological Principles for D2.2:**
1. **Exact neighborhood name matching:** Exact character-for-character matches between `listings_clean` (`neighbourhood_cleansed`) and `walkscore_clean` (`neighborhood`) are used for the 73 matching neighborhoods (2,733 listings); whether the two sources use identical neighborhood boundaries is not verified.
2. **Zero speculative geographic assumptions:** We do not rely on unverified external definitions, proximity assumptions, or assumed sub-district nesting.
3. **Unverified remaining correspondences:** For the 14 non-matching neighborhood names (1,085 listings), correspondences with the Walk Score table are not verified. Consequently, all 14 non-matching neighborhoods are classified strictly as **No confirmed correspondence**.
4. **Preservation of empirical missingness:** All 1,085 non-matching listings remain unmapped (`<NA>`). No synthetic imputation (mean/median) is applied.

### 6.1 Step 1: Identification of the 14 Unmapped Neighborhoods

We extract all 14 unmapped Airbnb neighborhoods, calculating listing counts, sample proportions, mean listing latitude and longitude, and parent groups directly from `listings_clean`.


In [8]:
# Extract and profile the 14 unmapped Airbnb neighborhoods
airbnb_hoods = set(listings_clean['neighbourhood_cleansed'].unique())
ws_hoods = set(walkscore_clean['neighborhood'].unique())
exact_matches = airbnb_hoods.intersection(ws_hoods)
unmatched_airbnb = sorted(list(airbnb_hoods - ws_hoods))

df_unmatched = (
    listings_clean[listings_clean['neighbourhood_cleansed'].isin(unmatched_airbnb)]
    .groupby(['neighbourhood_cleansed', 'neighbourhood_group_cleansed'])
    .agg(
        Listings=('id', 'count'),
        Mean_Listing_Latitude=('latitude', 'mean'),
        Mean_Listing_Longitude=('longitude', 'mean')
    )
    .reset_index()
    .rename(columns={
        'neighbourhood_cleansed': 'Airbnb Neighborhood',
        'neighbourhood_group_cleansed': 'Parent Group',
        'Mean_Listing_Latitude': 'Mean Listing Latitude',
        'Mean_Listing_Longitude': 'Mean Listing Longitude'
    })
    .sort_values(by='Listings', ascending=False)
)
df_unmatched['Pct of Dataset'] = (df_unmatched['Listings'] / len(listings_clean) * 100).round(2).astype(str) + '%'
df_unmatched['Mean Listing Latitude'] = df_unmatched['Mean Listing Latitude'].round(4)
df_unmatched['Mean Listing Longitude'] = df_unmatched['Mean Listing Longitude'].round(4)

assert len(df_unmatched) == 14, f'Expected 14 unmatched neighborhoods, got {len(df_unmatched)}'
assert df_unmatched['Listings'].sum() == 1085, f'Expected 1,085 listings, got {df_unmatched["Listings"].sum()}'

print(f'Identified {len(df_unmatched)} unmapped neighborhoods representing {df_unmatched["Listings"].sum():,} listings (28.42% of total).')
display(df_unmatched)


Identified 14 unmapped neighborhoods representing 1,085 listings (28.42% of total).


,Airbnb Neighborhood,Parent Group,Listings,Mean Listing Latitude,Mean Listing Longitude,Pct of Dataset
2,Broadway,Capitol Hill,397,47.6213,-122.3212,10.4%
8,Minor,Central Area,135,47.6092,-122.3095,3.54%
11,Stevens,Capitol Hill,119,47.6241,-122.3064,3.12%
4,Central Business District,Downtown,103,47.6094,-122.3355,2.7%
0,Adams,Ballard,70,47.6715,-122.3865,1.83%
7,Mann,Central Area,67,47.6143,-122.2994,1.75%
13,West Woodland,Ballard,64,47.6691,-122.3668,1.68%
1,Alki,West Seattle,42,47.5751,-122.4080,1.1%
10,Pike-Market,Downtown,28,47.6091,-122.3408,0.73%
12,Wedgwood,Other neighborhoods,18,47.6901,-122.2924,0.47%


### 6.2 Steps 2 & 3: Investigation & Decision Classification

Each of the 14 non-matching neighborhood names is documented against the local 80-row Walk Score dataset (`seattle_walkscore_raw.csv`).

In this project, exact neighborhood name matches are used; remaining correspondences are not verified, and we refrain from making unverified assumptions regarding sub-district relationships or neighborhood nesting.

**Classification Decision:**
All 14 non-matching neighborhoods are classified strictly as **`No confirmed correspondence`**. No candidate substitutions are adopted into the crosswalk.


In [9]:
# Construct Decision Classification Table for all 14 unmapped neighborhoods
# Strictly objective: no unverified external definitions, no proximity claims, no assumed nesting
unmatched_hoods_list = sorted(list(unmatched_airbnb))

classification_records = []
for hood in unmatched_hoods_list:
    cnt = int((listings_clean['neighbourhood_cleansed'] == hood).sum())
    parent_grp = listings_clean.loc[listings_clean['neighbourhood_cleansed'] == hood, 'neighbourhood_group_cleansed'].iloc[0]
    classification_records.append({
        'Airbnb Neighborhood': hood,
        'Parent Group': parent_grp,
        'Listings': cnt,
        'Walk Score Match': 'None',
        'Status': 'No confirmed correspondence',
        'Empirical Finding': f"Name '{hood}' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified."
    })

df_classification = pd.DataFrame(classification_records).sort_values(by='Listings', ascending=False).reset_index(drop=True)
display(df_classification)

# Status distribution table (dynamically calculated)
status_agg = (
    df_classification.groupby('Status')
    .agg(
        Neighborhoods=('Airbnb Neighborhood', 'count'),
        Listings=('Listings', 'sum')
    )
    .reset_index()
)
status_agg['Pct of Unmapped Listings (1,085)'] = (status_agg['Listings'] / df_classification['Listings'].sum() * 100).round(2).astype(str) + '%'
status_agg['Pct of Unmapped Neighborhoods (14)'] = (status_agg['Neighborhoods'] / len(df_classification) * 100).round(2).astype(str) + '%'
status_agg['Pct of Total Sample (3,818)'] = (status_agg['Listings'] / len(listings_clean) * 100).round(2).astype(str) + '%'
display(status_agg)


,Airbnb Neighborhood,Parent Group,Listings,Walk Score Match,Status,Empirical Finding
0,Broadway,Capitol Hill,397,None,No confirmed correspondence,Name 'Broadway' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
1,Minor,Central Area,135,None,No confirmed correspondence,Name 'Minor' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
2,Stevens,Capitol Hill,119,None,No confirmed correspondence,Name 'Stevens' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
3,Central Business District,Downtown,103,None,No confirmed correspondence,Name 'Central Business District' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
4,Adams,Ballard,70,None,No confirmed correspondence,Name 'Adams' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
5,Mann,Central Area,67,None,No confirmed correspondence,Name 'Mann' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
6,West Woodland,Ballard,64,None,No confirmed correspondence,Name 'West Woodland' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
7,Alki,West Seattle,42,None,No confirmed correspondence,Name 'Alki' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
8,Pike-Market,Downtown,28,None,No confirmed correspondence,Name 'Pike-Market' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.
9,Wedgwood,Other neighborhoods,18,None,No confirmed correspondence,Name 'Wedgwood' has no exact match in seattle_walkscore_raw.csv; remaining neighborhood correspondences are not verified.


,Status,Neighborhoods,Listings,"Pct of Unmapped Listings (1,085)",Pct of Unmapped Neighborhoods (14),"Pct of Total Sample (3,818)"
0,No confirmed correspondence,14,1085,100.0%,100.0%,28.42%


### 6.3 Step 4: Crosswalk Construction & Many-to-One Cardinality Verification

In accordance with our strict protocol:
- **No unverified correspondences are accepted.**
- The confirmed crosswalk dictionary is empty: `confirmed_crosswalk = {}`.
- All 14 non-matching neighborhoods (1,085 listings) resolve to `<NA>`.

**Relational Verification:**
- The relationship between listings and Walk Score is **Many-to-One (`m:1`)** (many listings map to a single neighborhood).
- We test this relationship via a simulated left join, verifying that listing IDs remain unique, the sample size remains exactly 3,818 rows, and zero Cartesian duplication occurs.


In [10]:
# Empty confirmed crosswalk: only exact neighborhood name matches are used; remaining correspondences are not verified
confirmed_crosswalk = {}

def resolve_walkscore_neighborhood(cleansed_name: str) -> str:
    """Return mapped Walk Score neighborhood name if exact match; else NA."""
    if cleansed_name in exact_matches:
        return cleansed_name
    elif cleansed_name in confirmed_crosswalk:
        return confirmed_crosswalk[cleansed_name]
    return pd.NA

mapped_series = listings_clean['neighbourhood_cleansed'].apply(resolve_walkscore_neighborhood).astype('string')

# Cardinality and join validation (Many-to-One relationship)
assert len(mapped_series) == len(listings_clean), 'Series length altered!'
assert mapped_series.dropna().isin(ws_hoods).all(), 'Found mapped key not in Walk Score dataset!'

# Simulated left join to verify many-to-one relationship and absence of row multiplication
df_join_test = pd.DataFrame({
    'listing_id': listings_clean['id'],
    'neighbourhood_cleansed': listings_clean['neighbourhood_cleansed'],
    'ws_neighborhood_key': mapped_series
}).merge(
    walkscore_clean[['neighborhood', 'walk_score', 'transit_score', 'bike_score']],
    left_on='ws_neighborhood_key',
    right_on='neighborhood',
    how='left'
)

# Rigorous relational assertions
assert len(df_join_test) == len(listings_clean), f'Join created duplicate rows! Expected {len(listings_clean)}, got {len(df_join_test)}'
assert df_join_test['listing_id'].is_unique, 'Listing IDs are not unique after join!'
assert df_join_test['walk_score'].notna().sum() == 2733, f'Expected exactly 2,733 matched listings, got {df_join_test["walk_score"].notna().sum()}'
assert df_join_test['walk_score'].isna().sum() == 1085, f'Expected exactly 1,085 unmapped listings, got {df_join_test["walk_score"].isna().sum()}'

print(f'Confirmed Crosswalk Dictionary: {confirmed_crosswalk} (0 candidate substitutions accepted)')
print(f'✓ Many-to-One (m:1) relational structure confirmed: exactly {len(df_join_test):,} rows preserved.')
print(f'✓ Listing ID uniqueness verified: {df_join_test["listing_id"].is_unique} (0 duplicate rows created).')
print(f'✓ Exactly 2,733 listings resolved via exact name match; exactly 1,085 listings remain unmapped (<NA>).')


Confirmed Crosswalk Dictionary: {} (0 candidate substitutions accepted)
✓ Many-to-One (m:1) relational structure confirmed: exactly 3,818 rows preserved.
✓ Listing ID uniqueness verified: True (0 duplicate rows created).
✓ Exactly 2,733 listings resolved via exact name match; exactly 1,085 listings remain unmapped (<NA>).


### 6.4 Step 5: Post-Mapping Coverage & Quality Audit

We quantify Walk Score coverage using fully dynamic calculations derived from `mapped_series` and `df_classification`. Because zero additional correspondences were confirmed, the coverage remains identical to the baseline exact string matches.


In [11]:
# Construct post-mapping Walk Score coverage summary table dynamically from DataFrames
n_exact_hoods = len(exact_matches)
n_exact_listings = int(mapped_series.notna().sum())
n_unmapped_hoods = len(unmatched_airbnb)
n_unmapped_listings = int(mapped_series.isna().sum())
total_listings = len(listings_clean)

coverage_records = [
    {
        'Metric / Category': 'Exact Name String Matches',
        'Neighborhoods': n_exact_hoods,
        'Listings': n_exact_listings,
        'Share of Sample': f"{n_exact_listings / total_listings:.2%}",
        'Methodological Status': 'Exact character-for-character name match used; neighborhood boundary equivalence is not verified.'
    },
    {
        'Metric / Category': 'Confirmed Crosswalk Additions',
        'Neighborhoods': len(confirmed_crosswalk),
        'Listings': 0,
        'Share of Sample': '0.00%',
        'Methodological Status': '0 additions; only exact name matches are used and remaining correspondences are not verified.'
    },
    {
        'Metric / Category': 'Post-Mapping Total Matched Sample',
        'Neighborhoods': n_exact_hoods + len(confirmed_crosswalk),
        'Listings': n_exact_listings,
        'Share of Sample': f"{n_exact_listings / total_listings:.2%}",
        'Methodological Status': 'Coverage uses exact neighborhood name matches only (2,733 listings across 73 neighborhoods).'
    },
    {
        'Metric / Category': 'No Confirmed Correspondence (Unmapped)',
        'Neighborhoods': n_unmapped_hoods,
        'Listings': n_unmapped_listings,
        'Share of Sample': f"{n_unmapped_listings / total_listings:.2%}",
        'Methodological Status': 'All 14 non-matching neighborhoods preserved strictly as missing values (NaN); zero synthetic imputation.'
    }
]

df_coverage = pd.DataFrame(coverage_records)
display(df_coverage)

# Assertions for D2.2 boundaries
assert len(listings_clean) == 3818, 'Listings row count altered in D2.2!'
assert len(walkscore_clean) == 80, 'Walk Score row count altered in D2.2!'

print('=== D2.2 WALK SCORE MAPPING VALIDATION SUMMARY ===')
print(f'✓ Baseline exact name matches: {n_exact_listings:,} listings across {n_exact_hoods} neighborhoods ({n_exact_listings/total_listings:.2%}).')
print('  (Note: exact name matching does not verify neighborhood boundary equivalence).')
print('✓ Crosswalk additions: 0 listings across 0 neighborhoods (crosswalk dictionary is empty).')
print(f"✓ Exactly {n_unmapped_hoods} neighborhoods ({n_unmapped_listings:,} listings, {n_unmapped_listings/total_listings:.2%}) retain status 'No confirmed correspondence'.")
print('✓ Many-to-One (m:1) join verified: exactly 3,818 unique listings preserved (zero Cartesian duplication).')
print('✓ Zero rows dropped; zero columns dropped; zero median/mean imputation performed.')
print('✓ D2.2 complete: exact neighborhood name matches used; remaining correspondences are not verified.')


=== D2.2 WALK SCORE MAPPING VALIDATION SUMMARY ===
✓ Baseline exact name matches: 2,733 listings across 73 neighborhoods (71.58%).
  (Note: exact name matching does not verify neighborhood boundary equivalence).
✓ Crosswalk additions: 0 listings across 0 neighborhoods (crosswalk dictionary is empty).
✓ Exactly 14 neighborhoods (1,085 listings, 28.42%) retain status 'No confirmed correspondence'.
✓ Many-to-One (m:1) join verified: exactly 3,818 unique listings preserved (zero Cartesian duplication).
✓ Zero rows dropped; zero columns dropped; zero median/mean imputation performed.
✓ D2.2 complete: exact neighborhood name matches used; remaining correspondences are not verified.


,Metric / Category,Neighborhoods,Listings,Share of Sample,Methodological Status
0,Exact Name String Matches,73,2733,71.58%,Exact character-for-character name match used; neighborhood boundary equivalence is not verified.
1,Confirmed Crosswalk Additions,0,0,0.00%,0 additions; only exact name matches are used and remaining correspondences are not verified.
2,Post-Mapping Total Matched Sample,73,2733,71.58%,"Coverage uses exact neighborhood name matches only (2,733 listings across 73 neighborhoods)."
3,No Confirmed Correspondence (Unmapped),14,1085,28.42%,All 14 non-matching neighborhoods preserved strictly as missing values (NaN); zero synthetic imputation.


## 7. Integrated Dataset Assembly (D2.3)

In this phase, we transition from cleaning (D2.1) and taxonomic alignment (D2.2) to the assembly of analysis-ready datasets. We construct two processed, integrated datasets in memory:
1. **Listing-Level Dataset (`df_listings_clean`):** Exactly 1 row per listing (N = 3,818). Integrates cleaned property attributes with Walk Score mobility metrics matched by exact name string equality (neighborhood boundary equivalence is not verified) and listing-level review summary metrics aggregated from `reviews.csv`.
2. **Calendar-Day Panel Dataset (`df_calendar_clean`):** Exactly 1 row per listing-day across the full calendar span (2016-01-04 to 2017-01-02, N = 1,393,570 listing-days across 365 calendar dates). Integrates cleaned daily calendar states (availability and prices) with daily holiday and citywide event indicators aggregated from `seattle_holidays_events_2016.csv`.

**Core Methodological Principles for D2.3:**
- **Zero Row Drops & Zero Column Drops:** All 3,818 listings and all 1,393,570 calendar listing-days are preserved. High-missingness columns (`license`, `square_feet`) remain in the dataset with documented analytical exclusion rationale.
- **Zero Synthetic Imputation:** All 1,085 listings in the 14 unmapped neighborhoods retain missing Walk Scores (`<NA>`). We do not apply mean, median, or proxy imputation.
- **Event Pre-Aggregation by Date:** To eliminate any possibility of Cartesian row multiplication, event records are aggregated to strictly 1 row per unique date before joining with the daily calendar. Dates with multiple concurrent events (such as Labor Day and PAX West on 2016-09-05) are preserved via structured concatenations and the `event_record_count` metric (which counts source records per date, not necessarily individual events).
- **Review Aggregation & Text Isolation:** Listing-level review counts and dates from `reviews.csv` are aggregated into non-colliding column names (`rev_*`). Raw review comments are preserved separately in `reviews_clean` for NLP text and topic modeling (Rett Kinsey).
- **Temporal Documentation & Leakage Prevention:** We preserve explicit timestamps across all records. Rather than making categorical claims of 'zero leakage', we explicitly document temporal boundaries: forward-looking calendar availability and prices across the full calendar span (2016-01-04 to 2017-01-02) are isolated from static listing metadata, allowing downstream modeling (Marvin Pearson) to establish task-specific prediction cutoffs.
- **Process Separation:** Full independent audit is reserved for Phase D2.4; file export to `data/processed/` is reserved for Phase D2.5. Datasets remain held in memory.


In [12]:
# Step 1: Listing-level review feature aggregation from reviews_clean
reviews_summary = reviews_clean.groupby('listing_id').agg(
    rev_count_from_reviews=('id', 'count'),
    rev_first_date_from_reviews=('date', 'min'),
    rev_last_date_from_reviews=('date', 'max'),
    rev_unique_reviewers=('reviewer_id', 'nunique'),
    rev_avg_comment_length=('comments', lambda s: s.str.len().mean())
).reset_index()

reviews_summary['rev_avg_comment_length'] = reviews_summary['rev_avg_comment_length'].round(1)

# Validate against listings_clean baseline
assert len(reviews_summary) == 3191, f'Expected 3,191 listings with reviews, got {len(reviews_summary)}'
assert reviews_summary['listing_id'].isin(listings_clean['id']).all(), 'Found review listing_id not in listings_clean!'

# Display review aggregation audit
print(f'Aggregated review metrics for {len(reviews_summary):,} listings from reviews_clean ({len(reviews_clean):,} total reviews).')
print(f'Listings without reviews in reviews_clean: {len(listings_clean) - len(reviews_summary):,} (16.42% of total sample).')
display(reviews_summary.head(5))


Aggregated review metrics for 3,191 listings from reviews_clean (84,849 total reviews).
Listings without reviews in reviews_clean: 627 (16.42% of total sample).


,listing_id,rev_count_from_reviews,rev_first_date_from_reviews,rev_last_date_from_reviews,rev_unique_reviewers,rev_avg_comment_length
0,4291,35,2013-07-01,2015-10-18,35,380.6
1,5682,297,2010-03-21,2015-12-14,289,376.4
2,6606,52,2009-07-17,2015-12-28,51,384.9
3,7369,40,2009-06-07,2012-03-04,39,239.8
4,9419,79,2010-07-30,2015-12-07,78,545.6


### 7.2 Step 2: Calendar Holiday & Event Aggregation by Date

Before merging holiday and event indicators with the 1.39 million row calendar, we aggregate `holidays_clean` strictly to **one row per unique calendar date**.

**Methodological Justification:**
- In `seattle_holidays_events_2016.csv`, events are recorded across 19 unique calendar dates. On dates where multiple events coincide (e.g., Labor Day and PAX West on September 5, 2016), aggregating by date guarantees $m:1$ cardinality between calendar listing-days and events, preventing Cartesian row multiplication.
- We construct structured aggregation fields (`event_names`, `event_types`, `event_neighborhoods`, `event_record_count`, `is_holiday`) that preserve the full information from coincident events. The `event_record_count` metric counts source records per date, not necessarily individual events.
- Exactly 18 of the 19 event dates fall within the calendar span (2016-01-04 to 2017-01-02); exactly 1 event date (`2016-01-01`, New Year's Day) precedes the calendar start date.


In [13]:
# Step 2: Aggregate holiday and event records to strictly 1 row per unique date
# Note: event_record_count counts source records per date, not necessarily individual events
events_by_date = holidays_clean.groupby('date').agg(
    event_record_count=('event_name', 'count'),
    event_names=('event_name', lambda s: '; '.join(s)),
    event_types=('type', lambda s: '; '.join(s.unique())),
    event_neighborhoods=('neighborhood_focus', lambda s: '; '.join(s.unique())),
    is_holiday=('type', lambda s: s.str.contains('Federal Holiday').any())
).reset_index()

assert events_by_date['date'].is_unique, 'Date key in events_by_date is not unique!'
assert len(events_by_date) == 19, f'Expected 19 unique event dates, got {len(events_by_date)}'

cal_dates = set(calendar_clean['date'].unique())
ev_dates = set(events_by_date['date'].unique())
matched_dates = ev_dates.intersection(cal_dates)
unmatched_dates = ev_dates - cal_dates

print(f'Total event dates in source: {len(events_by_date)} (all strictly unique).')
print(f'Event dates within calendar timeline (2016-01-04 to 2017-01-02): {len(matched_dates)}')
print(f'Event dates outside calendar timeline: {len(unmatched_dates)} ({[d.strftime("%Y-%m-%d") for d in unmatched_dates]})')
print(f'Expected matching calendar rows: {len(matched_dates)} dates × {len(listings_clean):,} listings = {len(matched_dates) * len(listings_clean):,} rows.')
display(events_by_date)


Total event dates in source: 19 (all strictly unique).
Event dates within calendar timeline (2016-01-04 to 2017-01-02): 18
Event dates outside calendar timeline: 1 (['2016-01-01'])
Expected matching calendar rows: 18 dates × 3,818 listings = 68,724 rows.


,date,event_record_count,event_names,event_types,event_neighborhoods,is_holiday
0,2016-01-01,1,New Year's Day,Federal Holiday,Citywide,True
1,2016-01-18,1,Martin Luther King Jr. Day,Federal Holiday,Citywide,True
2,2016-02-15,1,Washington's Birthday (Presidents' Day),Federal Holiday,Citywide,True
3,2016-05-30,1,Memorial Day,Federal Holiday,Citywide,True
4,2016-06-25,1,Seattle PrideFest,Seattle Event,Capitol Hill,False
5,2016-06-26,1,Seattle Pride Parade,Seattle Event,Downtown / Capitol Hill,False
6,2016-07-04,1,Independence Day,Federal Holiday,Citywide,True
7,2016-08-05,1,Seafair Weekend,Seattle Festival,Lake Washington / Rainier Valley,False
8,2016-08-06,1,Seafair Weekend & Blue Angels,Seattle Festival,Lake Washington / Rainier Valley,False
9,2016-08-07,1,Seafair Weekend & Blue Angels,Seattle Festival,Lake Washington / Rainier Valley,False


### 7.3 Step 3: Assembly of Listing-Level Analytical Dataset (`df_listings_clean`)

We construct `df_listings_clean` by joining `listings_clean` with:
1. Walk Score mobility metrics matched by exact neighborhood name string equality (`neighbourhood_cleansed == neighborhood`); neighborhood boundary equivalence is not verified.
2. Structural review summary aggregates on `id == listing_id`.

**Relational Guarantees:**
- Exactly 3,818 rows preserved; `id` is verified unique.
- All 92 baseline columns are retained, including documented exclusion candidates (`license`, `square_feet`).
- Walk Score coverage: exactly 2,733 listings matched (71.58%); exactly 1,085 listings retain `<NA>` (28.42%).
- Review aggregates: exactly 3,191 listings enriched; exactly 627 listings with 0 reviews retain `<NA>`.


In [14]:
# Step 3: Assemble df_listings_clean via left joins with Walk Score and Reviews summary
ws_features = walkscore_clean[['neighborhood', 'walk_score', 'transit_score', 'bike_score', 'population', 'rank']].rename(
    columns={'population': 'ws_population', 'rank': 'ws_rank'}
)

df_listings_clean = listings_clean.merge(
    ws_features, left_on='neighbourhood_cleansed', right_on='neighborhood', how='left'
).drop(columns=['neighborhood'])

df_listings_clean = df_listings_clean.merge(
    reviews_summary, left_on='id', right_on='listing_id', how='left'
).drop(columns=['listing_id'])

# Relational and integrity assertions
assert len(df_listings_clean) == 3818, f'Row count changed! Expected 3,818, got {len(df_listings_clean)}'
assert df_listings_clean['id'].is_unique, 'Primary key id is not unique!'
assert df_listings_clean['walk_score'].notna().sum() == 2733, 'Walk Score matched count distorted!'
assert df_listings_clean['walk_score'].isna().sum() == 1085, 'Walk Score missing count distorted!'
assert df_listings_clean['rev_count_from_reviews'].notna().sum() == 3191, 'Review count matched count distorted!'
assert df_listings_clean['rev_count_from_reviews'].isna().sum() == 627, 'Review count missing count distorted!'

print(f'✓ df_listings_clean assembled: {df_listings_clean.shape[0]:,} rows × {df_listings_clean.shape[1]} columns.')
print(f'✓ Primary key uniqueness: id.is_unique = {df_listings_clean["id"].is_unique} (0 duplicate rows).')
print(f'✓ Walk Score coverage preserved: 2,733 matched (71.58%), 1,085 unmapped (<NA>, 28.42%).')
print(f'✓ Review aggregates integrated: 3,191 listings enriched, 627 unreviewed (<NA>, 16.42%).')
display(df_listings_clean[['id', 'name', 'neighbourhood_cleansed', 'walk_score', 'transit_score', 'bike_score', 'rev_count_from_reviews', 'rev_unique_reviewers', 'rev_avg_comment_length']].head(5))


✓ df_listings_clean assembled: 3,818 rows × 102 columns.
✓ Primary key uniqueness: id.is_unique = True (0 duplicate rows).
✓ Walk Score coverage preserved: 2,733 matched (71.58%), 1,085 unmapped (<NA>, 28.42%).
✓ Review aggregates integrated: 3,191 listings enriched, 627 unreviewed (<NA>, 16.42%).


,id,name,neighbourhood_cleansed,walk_score,transit_score,bike_score,rev_count_from_reviews,rev_unique_reviewers,rev_avg_comment_length
0,241032,Stylish Queen Anne Apartment,West Queen Anne,84.0,59.0,65.0,207.0,207.0,364.8
1,953595,Bright & Airy Queen Anne Apartment,West Queen Anne,84.0,59.0,65.0,43.0,42.0,464.3
2,3308979,New Modern House-Amazing water view,West Queen Anne,84.0,59.0,65.0,20.0,20.0,408.4
3,7421966,Queen Anne Chateau,West Queen Anne,84.0,59.0,65.0,NaN,NaN,NaN
4,278830,Charming craftsman 3 bdm house,West Queen Anne,84.0,59.0,65.0,38.0,38.0,273.1


### 7.4 Step 4: Assembly of Calendar-Day Panel Dataset (`df_calendar_clean`)

We construct `df_calendar_clean` by joining `calendar_clean` with the pre-aggregated `events_by_date` table on `date`.

**Design Decisions & Schema Integrity:**
- **Compact Panel Schema:** To avoid unnecessary memory expansion across 1.39 million rows, static property attributes are not merged into the daily calendar table. Listings metadata can be joined dynamically in memory via `listing_id` when required.
- **Indicator Imputation & Definition of `is_event`:** The `is_event` flag strictly denotes the **presence of a holiday or event record in our source dataset** (`seattle_holidays_events_2016.csv`). An `is_event == False` value signifies solely that no record exists in this source for that date, and does **not** prove the absence of real-world events. On non-event dates (347 calendar dates), event indicators are explicitly populated: `is_event = False`, `event_record_count = 0`, and `is_holiday = False`, preserving nullable `<NA>` for descriptive strings (`event_names`, `event_types`).
- **Calendar Invariant:** The exact 1-to-1 correspondence between unavailable days (`available == False`) and missing prices is verified preserved across all 459,028 occurrences.


In [15]:
# Step 4: Assemble df_calendar_clean via left join with pre-aggregated events_by_date
df_calendar_clean = calendar_clean.merge(events_by_date, on='date', how='left')

# Cleanly construct non-event indicator flags
# Note: is_event denotes presence of a record in source; False does not prove absence of real-world events
# Note: event_record_count counts source records per date, not necessarily individual events
df_calendar_clean['is_event'] = df_calendar_clean['event_record_count'].notna()
df_calendar_clean['event_record_count'] = df_calendar_clean['event_record_count'].fillna(0).astype('Int64')
df_calendar_clean['is_holiday'] = df_calendar_clean['is_holiday'].astype('boolean').fillna(False)

# Cardinality and consistency assertions
assert len(df_calendar_clean) == 1393570, f'Expected 1,393,570 rows, got {len(df_calendar_clean)}'
assert df_calendar_clean.duplicated(subset=['listing_id', 'date']).sum() == 0, 'Duplicate (listing_id, date) pairs detected!'
assert df_calendar_clean['is_event'].sum() == 18 * 3818, f'Expected 68,724 event rows, got {df_calendar_clean["is_event"].sum()}'
assert (df_calendar_clean['is_event'] == False).sum() == 347 * 3818, f'Expected 1,324,846 non-event rows, got {(df_calendar_clean["is_event"] == False).sum()}'
assert (df_calendar_clean['available'] == False).sum() == 459028, 'Unavailable day count changed!'
assert df_calendar_clean['price'].isna().sum() == 459028, 'Missing price count misaligned with availability!'

print(f'✓ df_calendar_clean assembled: {df_calendar_clean.shape[0]:,} rows × {df_calendar_clean.shape[1]} columns.')
print(f'✓ Key uniqueness verified: (listing_id, date) unique across all {len(df_calendar_clean):,} rows (0 duplicates).')
print(f'✓ Event alignment: 68,724 event rows (18 dates × 3,818 listings), 1,324,846 non-event rows (347 dates × 3,818 listings).')
print(f'✓ Structural integrity: 459,028 unavailable dates strictly coincide with 459,028 null prices.')
display(df_calendar_clean[df_calendar_clean['is_event']].head(5))


✓ df_calendar_clean assembled: 1,393,570 rows × 10 columns.
✓ Key uniqueness verified: (listing_id, date) unique across all 1,393,570 rows (0 duplicates).
✓ Event alignment: 68,724 event rows (18 dates × 3,818 listings), 1,324,846 non-event rows (347 dates × 3,818 listings).
✓ Structural integrity: 459,028 unavailable dates strictly coincide with 459,028 null prices.


,listing_id,date,available,price,event_record_count,event_names,event_types,event_neighborhoods,is_holiday,is_event
14,241032,2016-01-18,True,85.0,1,Martin Luther King Jr. Day,Federal Holiday,Citywide,True,True
42,241032,2016-02-15,True,85.0,1,Washington's Birthday (Presidents' Day),Federal Holiday,Citywide,True,True
147,241032,2016-05-30,True,85.0,1,Memorial Day,Federal Holiday,Citywide,True,True
173,241032,2016-06-25,True,85.0,1,Seattle PrideFest,Seattle Event,Capitol Hill,False,True
174,241032,2016-06-26,True,85.0,1,Seattle Pride Parade,Seattle Event,Downtown / Capitol Hill,False,True


### 7.5 Step 5: Post-Assembly Validation Summary

We conclude Phase D2.3 with an inventory audit and verification of methodological invariants across the assembled datasets.

**Temporal Boundaries & Data Leakage Documentation:**
- `df_listings_clean` captures static listing attributes and cumulative historical review metrics as of January 2016.
- `df_calendar_clean` captures daily forward-looking prices, availability states, and event records across the full calendar span (2016-01-04 to 2017-01-02).
- To prevent data leakage, prospective calendar aggregates across the full timeline (e.g., average forward price or unavailable rate from 2016-01-04 to 2017-01-02) have **not** been merged into static listing features.
- Downstream modeling tasks (Marvin Pearson) must formally specify observation cutoffs and prediction targets relative to these temporal anchors.

**Stage Boundary Status:**
- Full multi-dimensional integration validation is conducted in **Phase D2.4**.
- Export of processed datasets to `data/processed/` in CSV format takes place in **Phase D2.5**.
- All assembled DataFrames remain active in memory; raw CSV source files remain 100% untouched.


In [16]:
# Step 5: Post-assembly dataset inventory table and D2.3 validation summary
assembly_audit = pd.DataFrame([
    {
        'Dataset Name': 'df_listings_clean',
        'Entity Grain': 'One row per listing',
        'Primary Key': 'id',
        'Rows (N)': len(df_listings_clean),
        'Columns (P)': df_listings_clean.shape[1],
        'Key Uniqueness': df_listings_clean['id'].is_unique,
        'Integrity & Null Invariants': '3,818 rows; 1,085 null Walk Scores (<NA>); 627 unreviewed listings (<NA>)'
    },
    {
        'Dataset Name': 'df_calendar_clean',
        'Entity Grain': 'One row per listing-day',
        'Primary Key': '(listing_id, date)',
        'Rows (N)': len(df_calendar_clean),
        'Columns (P)': df_calendar_clean.shape[1],
        'Key Uniqueness': df_calendar_clean.duplicated(subset=['listing_id', 'date']).sum() == 0,
        'Integrity & Null Invariants': '1,393,570 rows; 459,028 null prices strictly matching available==False; 68,724 event rows'
    },
    {
        'Dataset Name': 'reviews_clean',
        'Entity Grain': 'One row per review',
        'Primary Key': 'id',
        'Rows (N)': len(reviews_clean),
        'Columns (P)': reviews_clean.shape[1],
        'Key Uniqueness': reviews_clean['id'].is_unique,
        'Integrity & Null Invariants': '84,849 rows; preserved untouched for NLP text analysis (Rett Kinsey)'
    }
])

display(assembly_audit)

print('=== D2.3 DATASET ASSEMBLY VALIDATION SUMMARY ===')
print('✓ Both processed datasets (df_listings_clean, df_calendar_clean) successfully assembled in memory.')
print('✓ reviews_clean preserved independently for NLP text modeling (Rett Kinsey).')
print('✓ Zero row drops and zero column drops across all source tables.')
print('✓ Zero synthetic imputation: Walk Score missingness preserved strictly as <NA> for all 1,085 listings.')
print('✓ Event records pre-aggregated by date: zero Cartesian row multiplication in calendar-day panel.')
print('✓ Temporal boundaries documented: future calendar metrics isolated from static listing features.')
print('✓ Zero files exported to disk (export deferred to Phase D2.5 following Phase D2.4 validation).')
print('✓ Raw CSV files remain 100% untouched and unchanged.')


,Dataset Name,Entity Grain,Primary Key,Rows (N),Columns (P),Key Uniqueness,Integrity & Null Invariants
0,df_listings_clean,One row per listing,id,3818,102,True,"3,818 rows; 1,085 null Walk Scores (<NA>); 627..."
1,df_calendar_clean,One row per listing-day,"(listing_id, date)",1393570,10,True,"1,393,570 rows; 459,028 null prices strictly m..."
2,reviews_clean,One row per review,id,84849,6,True,"84,849 rows; preserved untouched for NLP text ..."


=== D2.3 DATASET ASSEMBLY VALIDATION SUMMARY ===
✓ Both processed datasets (df_listings_clean, df_calendar_clean) successfully assembled in memory.
✓ reviews_clean preserved independently for NLP text modeling (Rett Kinsey).
✓ Zero row drops and zero column drops across all source tables.
✓ Zero synthetic imputation: Walk Score missingness preserved strictly as <NA> for all 1,085 listings.
✓ Event records pre-aggregated by date: zero Cartesian row multiplication in calendar-day panel.
✓ Temporal boundaries documented: future calendar metrics isolated from static listing features.
✓ Zero files exported to disk (export deferred to Phase D2.5 following Phase D2.4 validation).
✓ Raw CSV files remain 100% untouched and unchanged.


## 8. Integration Validation (D2.4)

Phase D2.4 performs focused verification to confirm that multi-source integration in Phase D2.3 did not corrupt, distort, or misalign any data. We execute five targeted checks:
1. **Rows & Keys:** Confirm expected dataset dimensions, primary key uniqueness and non-nullness across all tables (`listings_clean`, `calendar_clean`, `reviews_clean`), zero orphan records, and complete 365-day balance per listing.
2. **Data Preservation:** Confirm that all original cleaned columns in `listings_clean` and `calendar_clean` retain identical element-wise values in the assembled datasets (`equals == True`).
3. **Walk Score Fidelity:** Confirm that joined scores for the 2,733 matched listings match the source Walk Score records exactly, and all 1,085 unmapped listings remain strictly `<NA>`.
4. **Reviews Concordance:** Split audit confirming exact alignment for the 3,191 reviewed listings (`rev_count_from_reviews == number_of_reviews`, first/last review dates) and complete `<NA>` representation across all new review fields for the 627 unreviewed listings.
5. **Calendar & Events Alignment:** Confirm the 365-day timeline (2016-01-04 to 2017-01-02), exact element-wise date alignment for all 68,724 event rows (18 dates × 3,818 listings) and 38,180 federal holiday rows (10 dates × 3,818 listings) with 0 date mismatches, and 100% alignment between missing prices and unavailable dates (459,028 rows).

All checks enforce programmatically asserted invariants and are summarized in a single compact validation table.

In [17]:
# Phase D2.4: Five Core Integration Validation Checks

# --- Check 1: Rows & Keys ---
c1_l_shape = df_listings_clean.shape == (3818, 102)
c1_l_key_uniq = df_listings_clean['id'].is_unique
c1_l_key_notnull = df_listings_clean['id'].notna().all()

c1_c_shape = df_calendar_clean.shape == (1393570, 10)
c1_c_key_uniq = not df_calendar_clean.duplicated(subset=['listing_id', 'date']).any()
c1_c_key_notnull = df_calendar_clean['listing_id'].notna().all() and df_calendar_clean['date'].notna().all()

c1_c_ref = df_calendar_clean['listing_id'].isin(df_listings_clean['id']).all()
c1_c_balance = (df_calendar_clean.groupby('listing_id')['date'].count() == 365).all()

c1_r_shape = reviews_clean.shape == (84849, 6)
c1_r_key_uniq = reviews_clean['id'].is_unique
c1_r_key_notnull = reviews_clean['id'].notna().all()
c1_r_ref = reviews_clean['listing_id'].isin(df_listings_clean['id']).all()

assert all([
    c1_l_shape, c1_l_key_uniq, c1_l_key_notnull,
    c1_c_shape, c1_c_key_uniq, c1_c_key_notnull, c1_c_ref, c1_c_balance,
    c1_r_shape, c1_r_key_uniq, c1_r_key_notnull, c1_r_ref
]), 'Check 1 Failed!'

# --- Check 2: Data Preservation ---
c2_l_preserv = df_listings_clean[listings_clean.columns].equals(listings_clean)
c2_c_preserv = df_calendar_clean[calendar_clean.columns].equals(calendar_clean)

assert c2_l_preserv and c2_c_preserv, 'Check 2 Failed!'

# --- Check 3: Walk Score Fidelity ---
c3_matched_count = (df_listings_clean['walk_score'].notna()).sum() == 2733
c3_unmatched_count = (df_listings_clean['walk_score'].isna() & df_listings_clean['transit_score'].isna() & df_listings_clean['bike_score'].isna()).sum() == 1085

merged_ws_verify = df_listings_clean[df_listings_clean['walk_score'].notna()].merge(
    walkscore_clean, left_on='neighbourhood_cleansed', right_on='neighborhood', how='inner'
)
c3_fidelity = (
    (merged_ws_verify['walk_score_x'] == merged_ws_verify['walk_score_y']).all() and
    (merged_ws_verify['transit_score_x'] == merged_ws_verify['transit_score_y']).all() and
    (merged_ws_verify['bike_score_x'] == merged_ws_verify['bike_score_y']).all()
)

assert c3_matched_count and c3_unmatched_count and c3_fidelity, 'Check 3 Failed!'

# --- Check 4: Reviews Concordance ---
reviewed_mask = df_listings_clean['number_of_reviews'] > 0
unreviewed_mask = df_listings_clean['number_of_reviews'] == 0

c4_rev_count_match = (df_listings_clean.loc[reviewed_mask, 'rev_count_from_reviews'] == df_listings_clean.loc[reviewed_mask, 'number_of_reviews']).all()
c4_rev_first_match = (df_listings_clean.loc[reviewed_mask, 'rev_first_date_from_reviews'] == df_listings_clean.loc[reviewed_mask, 'first_review']).all()
c4_rev_last_match = (df_listings_clean.loc[reviewed_mask, 'rev_last_date_from_reviews'] == df_listings_clean.loc[reviewed_mask, 'last_review']).all()

rev_agg_cols = ['rev_count_from_reviews', 'rev_first_date_from_reviews', 'rev_last_date_from_reviews', 'rev_unique_reviewers', 'rev_avg_comment_length']
c4_unrev_all_na = df_listings_clean.loc[unreviewed_mask, rev_agg_cols].isna().all().all()

assert (reviewed_mask.sum() == 3191) and (unreviewed_mask.sum() == 627) and c4_rev_count_match and c4_rev_first_match and c4_rev_last_match and c4_unrev_all_na, 'Check 4 Failed!'

# --- Check 5: Calendar & Events Alignment ---
c5_dates = (
    df_calendar_clean['date'].min() == pd.Timestamp('2016-01-04') and
    df_calendar_clean['date'].max() == pd.Timestamp('2017-01-02') and
    df_calendar_clean['date'].nunique() == 365
)
c5_event_rows = df_calendar_clean['is_event'].sum() == (18 * 3818)
c5_holiday_rows = df_calendar_clean['is_holiday'].sum() == (10 * 3818)

# Element-wise date matching with source: 0 date misalignments
cal_dates_set = set(df_calendar_clean['date'])
ev_dates_source = set(events_by_date['date']).intersection(cal_dates_set)
holiday_dates_source = set(events_by_date.loc[events_by_date['is_holiday'], 'date']).intersection(cal_dates_set)

c5_event_date_match = (df_calendar_clean['is_event'] == df_calendar_clean['date'].isin(ev_dates_source)).all()
c5_holiday_date_match = (df_calendar_clean['is_holiday'] == df_calendar_clean['date'].isin(holiday_dates_source)).all()

c5_price_avail_align = ((df_calendar_clean['price'].isna()) == (~df_calendar_clean['available'])).all()
c5_unavail_count = (df_calendar_clean['available'] == False).sum() == 459028

assert (
    c5_dates and c5_event_rows and c5_holiday_rows and
    c5_event_date_match and c5_holiday_date_match and
    c5_price_avail_align and c5_unavail_count
), 'Check 5 Failed!'

# --- Construct Summary Validation Table ---
validation_summary = pd.DataFrame([
    {
        'Check #': '1',
        'Category': 'Rows & Keys',
        'Scope & Validation Criteria': 'Expected shapes, unique & non-null keys across all tables, zero orphan records, 365-day balance',
        'Observed Value': 'Listings (3,818x102, id unique); Cal (1,393,570x10); Rev (84,849x6, id unique); 0 orphans; 365 d/listing',
        'Expected Value': 'All 3 tables: exact shapes, unique & non-null PKs; 0 orphans; 365 days/listing',
        'Status': 'PASS'
    },
    {
        'Check #': '2',
        'Category': 'Data Preservation',
        'Scope & Validation Criteria': 'Original cleaned columns retain identical element-wise values after left joins',
        'Observed Value': 'listings_clean: equals=True; calendar_clean: equals=True (100% preserved)',
        'Expected Value': 'Both equals() == True across all 92 listing & 4 calendar columns',
        'Status': 'PASS'
    },
    {
        'Check #': '3',
        'Category': 'Walk Score Fidelity',
        'Scope & Validation Criteria': 'Joined scores match source table exactly; unmapped remain strictly <NA>',
        'Observed Value': 'Matched: 2,733 (100% score match); Unmapped: 1,085 (all 3 scores <NA>)',
        'Expected Value': 'Exactly 2,733 matched (100% match); exactly 1,085 unmapped (<NA>)',
        'Status': 'PASS'
    },
    {
        'Check #': '4',
        'Category': 'Reviews Concordance',
        'Scope & Validation Criteria': 'Split audit: 3,191 reviewed match counts/dates; 627 unreviewed strictly <NA>',
        'Observed Value': 'Reviewed: 3,191 (100% match on count/dates); Unreviewed: 627 (100% <NA>)',
        'Expected Value': '3,191 reviewed match exactly; 627 unreviewed have all 5 rev_* as <NA>',
        'Status': 'PASS'
    },
    {
        'Check #': '5',
        'Category': 'Calendar & Events',
        'Scope & Validation Criteria': 'Timeline range (365 d), 18 event & 10 holiday dates (0 date mismatches), price-availability 1:1 match',
        'Observed Value': '365 d (2016-01-04..2017-01-02); Events: 68,724 (0 date err); Holidays: 38,180 (0 date err); Unavail==Null: 459,028',
        'Expected Value': '365 dates; 68,724 event & 38,180 holiday rows on exact source dates; 459,028 null price',
        'Status': 'PASS'
    }
])

display(validation_summary)

print('=== D2.4 INTEGRATION VALIDATION RESULTS: ALL CHECKS PASSED ===')
print('✓ Check 1 (Rows & Keys): Expected shapes, PK uniqueness & non-nullness (Listings, Calendar, Reviews), referential integrity, and 365-day balance confirmed.')
print('✓ Check 2 (Data Preservation): All 92 listing and 4 calendar original columns 100% identical post-join.')
print('✓ Check 3 (Walk Score Fidelity): 2,733 matched scores match source; exactly 1,085 remain <NA>.')
print('✓ Check 4 (Reviews Concordance): 3,191 reviewed match baseline; 627 unreviewed have all rev_* as <NA>.')
print('✓ Check 5 (Calendar & Events): 365-day timeline, 0 date mismatches on events/holidays (68,724 & 38,180 rows), 459,028 null prices confirmed.')


,Check #,Category,Scope & Validation Criteria,Observed Value,Expected Value,Status
0,1,Rows & Keys,"Expected shapes, unique & non-null keys across all tables, zero orphan records, 365-day balance","Listings (3,818x102, id unique); Cal (1,393,570x10); Rev (84,849x6, id unique); 0 orphans; 365 d/listing","All 3 tables: exact shapes, unique & non-null PKs; 0 orphans; 365 days/listing",PASS
1,2,Data Preservation,Original cleaned columns retain identical element-wise values after left joins,listings_clean: equals=True; calendar_clean: equals=True (100% preserved),Both equals() == True across all 92 listing & 4 calendar columns,PASS
2,3,Walk Score Fidelity,Joined scores match source table exactly; unmapped remain strictly <NA>,"Matched: 2,733 (100% score match); Unmapped: 1,085 (all 3 scores <NA>)","Exactly 2,733 matched (100% match); exactly 1,085 unmapped (<NA>)",PASS
3,4,Reviews Concordance,"Split audit: 3,191 reviewed match counts/dates; 627 unreviewed strictly <NA>","Reviewed: 3,191 (100% match on count/dates); Unreviewed: 627 (100% <NA>)","3,191 reviewed match exactly; 627 unreviewed have all 5 rev_* as <NA>",PASS
4,5,Calendar & Events,"Timeline range (365 d), 18 event & 10 holiday dates (0 date mismatches), price-availability 1:1 match","365 d (2016-01-04..2017-01-02); Events: 68,724 (0 date err); Holidays: 38,180 (0 date err); Unavail==Null: 459,028","365 dates; 68,724 event & 38,180 holiday rows on exact source dates; 459,028 null price",PASS


=== D2.4 INTEGRATION VALIDATION RESULTS: ALL CHECKS PASSED ===
✓ Check 1 (Rows & Keys): Expected shapes, PK uniqueness & non-nullness (Listings, Calendar, Reviews), referential integrity, and 365-day balance confirmed.
✓ Check 2 (Data Preservation): All 92 listing and 4 calendar original columns 100% identical post-join.
✓ Check 3 (Walk Score Fidelity): 2,733 matched scores match source; exactly 1,085 remain <NA>.
✓ Check 4 (Reviews Concordance): 3,191 reviewed match baseline; 627 unreviewed have all rev_* as <NA>.
✓ Check 5 (Calendar & Events): 365-day timeline, 0 date mismatches on events/holidays (68,724 & 38,180 rows), 459,028 null prices confirmed.


## 9. Data Export & Documentation (D2.5)

Phase D2.5 finalizes the data cleaning and integration workflow by saving the three processed datasets to `data/processed/` as standard CSV files (`index=False`).

### 9.1 Data Serialization & Type Restoration Protocol
Standard CSV does not serialize pandas data types natively. When reloading processed datasets in downstream exploratory analysis (EDA) and predictive modeling notebooks, explicit type reconstruction is required:
- **Dates:** ISO-8601 strings (`YYYY-MM-DD`) must be parsed via `parse_dates=[...]` to restore `datetime64[ns]`.
- **Boolean Flags:** Binary indicator strings (`'True'` / `'False'` / empty) must be restored via `dtype={col: 'boolean'}` during read-time to ensure native nullable boolean support without post-read string type errors.
- **Nullable Integers:** Columns with valid integers and missing entries (`<NA>`) — such as Walk Score metrics and review counts — are restored to nullable integer `Int64` via `dtype={col: 'Int64'}`.

### 9.2 Processed Dataset Reloading Specification for Team Workstreams

```python
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("data/processed")

# 1. Listings Reload
DATE_COLS_LISTINGS = [
    'last_scraped', 'host_since', 'calendar_last_scraped',
    'first_review', 'last_review',
    'rev_first_date_from_reviews', 'rev_last_date_from_reviews'
]
BOOL_COLS_LISTINGS = [
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
    'is_location_exact', 'has_availability', 'requires_license',
    'instant_bookable', 'require_guest_profile_picture', 'require_guest_phone_verification'
]
INT_COLS_LISTINGS = [
    'walk_score', 'transit_score', 'bike_score', 'ws_population', 'ws_rank',
    'rev_count_from_reviews', 'rev_unique_reviewers'
]

dtypes_listings = {col: 'boolean' for col in BOOL_COLS_LISTINGS}
dtypes_listings.update({col: 'Int64' for col in INT_COLS_LISTINGS})

df_listings = pd.read_csv(
    PROCESSED_DIR / 'listings_clean.csv',
    parse_dates=DATE_COLS_LISTINGS,
    dtype=dtypes_listings
)

# 2. Calendar Reload
DATE_COLS_CALENDAR = ['date']
BOOL_COLS_CALENDAR = ['available', 'is_holiday', 'is_event']
INT_COLS_CALENDAR = ['event_record_count']

dtypes_calendar = {col: 'boolean' for col in BOOL_COLS_CALENDAR}
dtypes_calendar.update({col: 'Int64' for col in INT_COLS_CALENDAR})

df_calendar = pd.read_csv(
    PROCESSED_DIR / 'calendar_clean.csv',
    parse_dates=DATE_COLS_CALENDAR,
    dtype=dtypes_calendar
)

# 3. Reviews Reload
reviews_clean = pd.read_csv(
    PROCESSED_DIR / 'reviews_clean.csv',
    parse_dates=['date']
)
```


In [18]:
# Phase D2.5: Dataset Serialization & Full Readback Verification
from pandas.testing import assert_frame_equal

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export datasets to CSV
listings_path = PROCESSED_DIR / 'listings_clean.csv'
calendar_path = PROCESSED_DIR / 'calendar_clean.csv'
reviews_path = PROCESSED_DIR / 'reviews_clean.csv'

df_listings_clean.to_csv(listings_path, index=False)
df_calendar_clean.to_csv(calendar_path, index=False)
reviews_clean.to_csv(reviews_path, index=False)

# 2. Define explicit type restoration schemas
DATE_COLS_LISTINGS = [
    'last_scraped', 'host_since', 'calendar_last_scraped',
    'first_review', 'last_review',
    'rev_first_date_from_reviews', 'rev_last_date_from_reviews'
]
BOOL_COLS_LISTINGS = [
    'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
    'is_location_exact', 'has_availability', 'requires_license',
    'instant_bookable', 'require_guest_profile_picture', 'require_guest_phone_verification'
]
INT_COLS_LISTINGS = [
    'walk_score', 'transit_score', 'bike_score', 'ws_population', 'ws_rank',
    'rev_count_from_reviews', 'rev_unique_reviewers'
]

DATE_COLS_CALENDAR = ['date']
BOOL_COLS_CALENDAR = ['available', 'is_holiday', 'is_event']
INT_COLS_CALENDAR = ['event_record_count']

DATE_COLS_REVIEWS = ['date']

# 3. Perform full readback with read-time boolean and date parsing
dtypes_listings = {col: 'boolean' for col in BOOL_COLS_LISTINGS}
dtypes_listings.update({col: 'Int64' for col in INT_COLS_LISTINGS})

rb_listings = pd.read_csv(
    listings_path,
    parse_dates=DATE_COLS_LISTINGS,
    dtype=dtypes_listings
)

dtypes_calendar = {col: 'boolean' for col in BOOL_COLS_CALENDAR}
dtypes_calendar.update({col: 'Int64' for col in INT_COLS_CALENDAR})

rb_calendar = pd.read_csv(
    calendar_path,
    parse_dates=DATE_COLS_CALENDAR,
    dtype=dtypes_calendar
)

rb_reviews = pd.read_csv(
    reviews_path,
    parse_dates=DATE_COLS_REVIEWS
)

# 4. Comprehensive assertion verification across all reloaded datasets
# 4.1 Verify shapes and column order
assert rb_listings.shape == df_listings_clean.shape, f"Listings shape mismatch: {rb_listings.shape} vs {df_listings_clean.shape}"
assert list(rb_listings.columns) == list(df_listings_clean.columns), "Listings column order altered"
assert rb_calendar.shape == df_calendar_clean.shape, f"Calendar shape mismatch: {rb_calendar.shape} vs {df_calendar_clean.shape}"
assert list(rb_calendar.columns) == list(df_calendar_clean.columns), "Calendar column order altered"
assert rb_reviews.shape == reviews_clean.shape, f"Reviews shape mismatch: {rb_reviews.shape} vs {reviews_clean.shape}"
assert list(rb_reviews.columns) == list(reviews_clean.columns), "Reviews column order altered"

# 4.2 Verify primary keys and non-null constraints
assert rb_listings['id'].is_unique and rb_listings['id'].notna().all(), "Listings primary key corrupted"
assert not rb_calendar.duplicated(subset=['listing_id', 'date']).any(), "Calendar compound key corrupted"
assert rb_reviews['id'].is_unique and rb_reviews['id'].notna().all(), "Reviews primary key corrupted"

# 4.3 Verify 100% null position preservation across every column in all three datasets
assert (rb_listings.isna() == df_listings_clean.isna()).all().all(), "Listings null mask mismatch"
assert (rb_calendar.isna() == df_calendar_clean.isna()).all().all(), "Calendar null mask mismatch"
assert (rb_reviews.isna() == reviews_clean.isna()).all().all(), "Reviews null mask mismatch"

# 4.4 Verify complete review comment text preservation across all rows (no row subsetting)
assert (rb_reviews.loc[rb_reviews['comments'].notna(), 'comments'] == 
        reviews_clean.loc[reviews_clean['comments'].notna(), 'comments']).all(), "Review comments text mismatch"

# 4.5 Full DataFrame element-wise and numerical equality audits
assert_frame_equal(rb_reviews, reviews_clean, check_dtype=False)
assert_frame_equal(rb_calendar, df_calendar_clean, check_dtype=False, check_like=False)
assert_frame_equal(rb_listings, df_listings_clean, check_dtype=False, check_exact=False, rtol=1e-4, atol=1e-4)

# 5. Build compact export inventory summary table
export_inventory = pd.DataFrame([
    {
        'File Name': 'listings_clean.csv',
        'Entity Grain': 'One row per listing',
        'Rows (N)': f"{len(rb_listings):,}",
        'Columns (P)': rb_listings.shape[1],
        'File Size (MB)': f"{listings_path.stat().st_size / (1024**2):.2f}",
        'Readback Status': 'Export completed; readback checks passed'
    },
    {
        'File Name': 'calendar_clean.csv',
        'Entity Grain': 'One row per listing-day',
        'Rows (N)': f"{len(rb_calendar):,}",
        'Columns (P)': rb_calendar.shape[1],
        'File Size (MB)': f"{calendar_path.stat().st_size / (1024**2):.2f}",
        'Readback Status': 'Export completed; readback checks passed'
    },
    {
        'File Name': 'reviews_clean.csv',
        'Entity Grain': 'One row per review',
        'Rows (N)': f"{len(rb_reviews):,}",
        'Columns (P)': rb_reviews.shape[1],
        'File Size (MB)': f"{reviews_path.stat().st_size / (1024**2):.2f}",
        'Readback Status': 'Export completed; readback checks passed'
    }
])

display(export_inventory)

print("=== D2.5 EXPORT & VALIDATION RESULTS: COMPLETE ===")
print("✓ Full readback verification completed with read-time boolean and date parsing.")
print("✓ Full DataFrame equality (assert_frame_equal) verified across all 3 tables.")
print("✓ Complete null position mask (100% exact match) verified across all columns.")
print(f"✓ Review comments text 100% verified across all {len(reviews_clean):,} rows without subsetting.")
print("✓ Raw CSV files in data/ remain 100% untouched and unchanged.")


,File Name,Entity Grain,Rows (N),Columns (P),File Size (MB),Readback Status
0,listings_clean.csv,One row per listing,"3,818",102,16.42,Export completed; readback checks passed
1,calendar_clean.csv,One row per listing-day,"1,393,570",10,63.70,Export completed; readback checks passed
2,reviews_clean.csv,One row per review,"84,849",6,34.90,Export completed; readback checks passed


=== D2.5 EXPORT & VALIDATION RESULTS: COMPLETE ===
✓ Full readback verification completed with read-time boolean and date parsing.
✓ Full DataFrame equality (assert_frame_equal) verified across all 3 tables.
✓ Complete null position mask (100% exact match) verified across all columns.
✓ Review comments text 100% verified across all 84,849 rows without subsetting.
✓ Raw CSV files in data/ remain 100% untouched and unchanged.
